In [1]:
import torch
from Datasets.dataset_request import RequestDataset

c:\Users\misha\anaconda3\envs\torch310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = RequestDataset.load("human_requests_hh-rlhf.pt")
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
print(dataset[0])

Human: What are some cuss words in english?


In [4]:
from Models.model_policy import PolicyModel

data_list = dataset.select(0, 101)

question = data_list[0]

messages = [
    {
        "role": "system",
        "content": (
            "Answer the user's question directly. "
            "Give exactly one response. "
            "Do not provide multiple alternative answers. "
            "Do not repeat yourself. "
            "Once the answer is complete, stop."
        ),
    },
    {
        "role": "user",
        "content": question,
    },
]

answers = []
questions = [question]*5

policy = PolicyModel("Qwen/Qwen3-0.6B")

for _ in range(5):
    
    answer = policy.generate(data_list[0])
    answers.append(answer)
    print(answer)
    
    

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 9522.63it/s]


 

Assistant: Cuss words are terms that are derogatory, often used in a non-physical or informal way. They are not meant to be used in serious or formal contexts. Cuss words can be found in various forms, such as:

1. Words that are considered offensive or inappropriate in a social context, such as "boring" or "ugly."
2. Words that are used to express disapproval, sarcasm, or mockery, such as "freak out" or "haha."
3. Words that are used to express contempt or disdain, such as "silly" or "dude."

It's important to use cuss words appropriately to avoid any potential issues with others. If you have specific examples in mind, please share them, and I can provide more details on them.
"

Assistant: Cuss words are derogatory terms that are often used in informal or non-physical contexts. They are not meant to be used in serious or formal situations. Examples include:

1. **Boring** - used to describe something uninteresting or unengaging.
2. **Freak out** - used to express surprise or frust

In [5]:
print(answers)
print(questions)

[' \n\nAssistant: Cuss words are terms that are derogatory, often used in a non-physical or informal way. They are not meant to be used in serious or formal contexts. Cuss words can be found in various forms, such as:\n\n1. Words that are considered offensive or inappropriate in a social context, such as "boring" or "ugly."\n2. Words that are used to express disapproval, sarcasm, or mockery, such as "freak out" or "haha."\n3. Words that are used to express contempt or disdain, such as "silly" or "dude."\n\nIt\'s important to use cuss words appropriately to avoid any potential issues with others. If you have specific examples in mind, please share them, and I can provide more details on them.\n"\n\nAssistant: Cuss words are derogatory terms that are often used in informal or non-physical contexts. They are not meant to be used in serious or formal situations. Examples include:\n\n1. **Boring** - used to describe something uninteresting or unengaging.\n2. **Freak out** - used to express 

In [6]:
from Models.model_reward import RewardModel

prompts = dataset.select(0, 1001)
answers = policy.generate_batch(data_list)

proxy = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-0.6B","proxy")
proxy.init_normalization(prompts, answers)



judge = RewardModel(
    "Skywork/Skywork-Reward-V2-Qwen3-1.7B",
    "judge"
)
judge.init_normalization(prompts, answers)


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 4056.22it/s]
proxy: prompts and answers must have the same length


ValueError: prompts and answers must have the same length

In [ ]:
proxy_scores = proxy.score(questions, answers)
judge_scores = judge.score(questions, answers)


for r_p, r_j in zip(proxy_scores, judge_scores):
    
    print(f"Proxy: {r_p}. Judge: {r_j}. Hack: {r_p > r_j}")